**LCEL Basics – Concept**
* LCEL stands for LangChain Expression Language. It's a declarative way to build chains by connecting components with the pipe (|) operator.

Key Ideas

**Runnable protocol:**
Every component (prompt template, model, parser, etc.) is a Runnable. A Runnable is an object that has methods like:

- invoke() – call once and wait for the final result.

- stream() – get tokens as they are generated.

- ainvoke() / astream() – async versions.

* Pipe (|) operator
Connects Runnables so the output of one becomes the input of the next.
Example: prompt | model | parser creates a sequence.

**RunnablePassthrough** is a LangChain component that passes its input forward without any changes.  
Use it when you need to keep the original input while other parts of the chain do their work, or when you want to add extra keys to the input.


Visual Diagram



<div style="font-size: 0.85em; width: 50%; margin: 0 auto;">
  <h3 style="color: #e6edf3;">🔗 Basic LCEL Chain</h3>
  <pre style="background-color: #041322; color: #e6edf3; padding: 12px; border-radius: 6px; border: 1px solid #30363d; font-family: 'Courier New', monospace; line-height: 1.5; overflow-x: auto;">
Input dict (e.g., {"topic": "RAG"})
    |
    v
[Prompt Template]   -- fills placeholders, returns messages
    |
    v
[Chat Model]        -- returns AIMessage
    |
    v
[Output Parser]     -- returns final output
  </pre>

  <h3 style="color: #e6edf3;">🧩 Using <code>RunnablePassthrough</code></h3>
  <pre style="background-color: #041322; color: #e6edf3; padding: 12px; border-radius: 6px; border: 1px solid #30363d; font-family: 'Courier New', monospace; line-height: 1.5; overflow-x: auto;">
Input dict {"topic": "RAG", "user_id": "123"}
    |
    v
{RunnablePassthrough}  -- just passes the whole dict unchanged
  </pre>
</div>

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from dotenv import load_dotenv

In [2]:
# Load environment variables
load_dotenv

<function dotenv.main.load_dotenv(dotenv_path: Union[str, ForwardRef('os.PathLike[str]'), NoneType] = None, stream: Optional[IO[str]] = None, verbose: bool = False, override: bool = False, interpolate: bool = True, encoding: Optional[str] = 'utf-8') -> bool>

Step 1: Apply RunnablePassthrough 

In [3]:
# Create a RunnablePassthrough instance
passthrough = RunnablePassthrough()

# Invoke it with basic input
result = passthrough.invoke({
    'topic': 'RAG',
    'user_id': 'student_101'
})

# Get the output
print(result)
print(f'Type: {type(result)}')

{'topic': 'RAG', 'user_id': 'student_101'}
Type: <class 'dict'>


**RunnablePassthrough** simply returned the exact same dictionary we gave it, unchanged.

Step 3: Use RunnablePassthrough to Keep Original Input

**RunnableParallel** is a Runnable that executes multiple branches simultaneously and combines their outputs into a **dictionary**. Each branch can be a chain or a simple passthrough.

In [ ]:
# Build the chain using RunnableParallel

# Create the model and prompt
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', 'Explain {topic} in one sentence')
])

# Create the answer chain
chain = prompt | llm | StrOutputParser()

# Build a parallel chain that also keeps user_id
complete_chain = RunnableParallel(
    answer=chain,
    user_id=RunnablePassthrough()
)

# Invoke containing both topic and user_id
result = complete_chain.invoke({'topic': 'Machine learning', 'user_id': 'student101'})

print('Result')
print(result)
print(f'Type: {type(result)}')

Result
{'answer': 'Machine learning is a subset of artificial intelligence that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention.', 'user_id': {'topic': 'Machine learning', 'user_id': 'student101'}}
Type: <class 'dict'>


To Extract a Specific Field using lambda function

In [8]:
complete_chain_1 = RunnableParallel(
    answer=chain,
    user_id=lambda x: x['user_id']  # Extrack only user_id
)

# Invoke
result = complete_chain_1.invoke(
    {'topic': 'Machine learning', 'user_id': 'student_101'}
)

print('Result:')
print(result)
print('Type:', type(result))

Result:
{'answer': 'Machine learning is a subset of artificial intelligence that enables systems to learn from data, identify patterns, and make decisions with minimal human intervention.', 'user_id': 'student_101'}
Type: <class 'dict'>


**RunnableLambda – Concept**

* RunnableLambda: wraps a plain Python function and turns it into a Runnable so it can be used inside a chain with the pipe (|) operator.

It receives the output of the previous component.

It returns a value that becomes the input to the next component.

It is used for custom transformations, validation, logging, or any logic that doesn't fit into a prompt or model.

In [17]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', 'Explain {topic} in one sentence')
])

Step: Add a RunnableLambda to the Chain

In [18]:
# Define a custom function to transform the output
def result_to_upper(text):
    
    return text.upper() + '!!!'

# Wrap the function as a Runnable
wrap_function = RunnableLambda(result_to_upper)

# Build the chain: 
chain = prompt | llm | StrOutputParser() | wrap_function

# Invoke the chain
response = chain.invoke({'topic': 'Machine learning'})

print('Result:')
print(response)
print('Type:', type(response))

Result:
MACHINE LEARNING IS A SUBSET OF ARTIFICIAL INTELLIGENCE THAT ENABLES SYSTEMS TO LEARN FROM DATA, IDENTIFY PATTERNS, AND MAKE DECISIONS WITH MINIMAL HUMAN INTERVENTION.!!!
Type: <class 'str'>
